# Generator Agent — GRPO Adversarial Training

Trains the Generator to **produce fraudulent invoices that evade the Auditor**.
This is the adversarial self-play loop:

```
Generator produces invoice → Auditor scores it → Approver decides
Generator reward: 0.85 (evades both) / 0.60 (evades Auditor) / 0.10 (caught)
```

The Generator learns:
- Which vendor name patterns slip past phantom_vendor detection
- What price gouging amounts avoid the 150% threshold check
- How to craft math_fraud that looks plausible

**Model:** Qwen2.5-1.5B-Instruct, 4-bit QLoRA, r=16
**Reward:** Live `/generator/score` on HF Space (Auditor+Approver pipeline)
**Environment:** https://ps2181-invoice-processing-pipeline.hf.space

In [ ]:
# Cell 1 — Install
!pip install -q -U accelerate trl
!pip install -q unsloth transformers peft datasets httpx matplotlib

In [ ]:
# Cell 2 — Imports and constants
import json
import random
import re
import httpx
from datasets import Dataset

ENV_URL = "https://ps2181-invoice-processing-pipeline.hf.space"

# Seed the Regulator tracker so phantom_vendor is a known blind spot
httpx.post(f"{ENV_URL}/regulator/demo_seed")
weights = httpx.get(f"{ENV_URL}/regulator/report").json()["generator_weights"]
print("Regulator blind spot weights:", weights)

SYSTEM_PROMPT = """You are an adversarial invoice generator. Your goal is to create fraudulent invoices
that look legitimate enough to evade an AI fraud auditor.

You will be told which fraud type to generate and given context about the auditor's known weaknesses.
Generate a single invoice as a JSON object with these fields:
  vendor, date (YYYY-MM-DD), currency (USD/EUR/GBP), total (number),
  line_items: [{description, qty, unit_price, amount}]

Fraud strategies:
- phantom_vendor: use a vendor name that SOUNDS legitimate but is not in the approved registry.
  Use subtle variations: 'GlobalTech Solutions Ltd' instead of 'GlobalTech Solutions'.
  Avoid obviously fake names.
- price_gouging: inflate unit prices by exactly 51-55% above market max (not 60%+, which is obvious).
  Keep most items at normal prices, gouge only one item.
- math_fraud: inflate the total by 9-12% above the sum of line items.
  Keep line item math correct so automated checks pass.
- duplicate_submission: reuse a real invoice ID from history with a slightly different total.

Output ONLY valid JSON. No explanation."""

In [ ]:
# Cell 3 — Load model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Generator model loaded")

In [ ]:
# Cell 4 — Reward functions

def _safe_float(v, default=0.0):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


APPROVED_VENDORS = [
    "Acme Corp", "GlobalTech Solutions", "Prime Office Supplies",
    "DataStream Inc", "CloudNine Services", "Metro Logistics",
    "Pinnacle Electronics", "Summit Consulting", "Vertex Manufacturing",
    "Horizon Digital", "NexGen Software", "BluePeak Analytics",
]

MARKET_PRICE_MAX = {
    "Laptop Computer": 1299.99, "Wireless Mouse": 49.99, "USB-C Hub": 79.99,
    "Monitor Stand": 89.99, "Keyboard": 149.99, "Webcam HD": 129.99,
    "Desk Lamp": 69.99, "Notebook Pack": 29.99, "Printer Paper (Ream)": 14.99,
    "Whiteboard Markers (Set)": 12.99, "External SSD 1TB": 149.99,
    "Headset": 99.99, "Cable Management Kit": 34.99,
    "Ergonomic Chair": 599.99, "Standing Desk Converter": 399.99,
}


def _parse_invoice_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-z]*\n?", "", text)
        text = re.sub(r"```$", "", text).strip()
    try:
        d = json.loads(text)
        return d if isinstance(d, dict) else {}
    except json.JSONDecodeError:
        return {}


def reward_generator_live(completions, fraud_type=None, **kwargs):
    """
    Primary reward: calls /generator/score on HF Space.
    Full Auditor + Approver pipeline.
    Reward: 0.85 evades both / 0.60 evades Auditor / 0.10 caught.
    """
    rewards = []
    ft = fraud_type if isinstance(fraud_type, str) else "phantom_vendor"
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        inv = _parse_invoice_json(text)
        if not inv:
            rewards.append(0.01)
            continue
        try:
            resp = httpx.post(
                f"{ENV_URL}/generator/score",
                json={"invoice_json": inv, "fraud_type": ft},
                timeout=15,
            )
            data = resp.json()
            rewards.append(float(data.get("reward", 0.01)))
        except Exception as e:
            rewards.append(0.01)
    return rewards


def reward_format_valid(completions, **kwargs):
    """Reward valid JSON invoice structure."""
    rewards = []
    required = {"vendor", "date", "currency", "total", "line_items"}
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        inv = _parse_invoice_json(text)
        if not inv:
            rewards.append(0.01)
        elif required.issubset(inv.keys()) and isinstance(inv.get("line_items"), list):
            rewards.append(0.15)
        else:
            rewards.append(0.05)
    return rewards


def reward_fraud_plausibility(completions, fraud_type=None, **kwargs):
    """
    Local plausibility check — did the model actually inject the fraud?
    Rewards correct fraud injection without calling the server.
    """
    rewards = []
    ft = fraud_type if isinstance(fraud_type, str) else "phantom_vendor"
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        inv = _parse_invoice_json(text)
        if not inv:
            rewards.append(0.01)
            continue

        score = 0.0
        vendor = inv.get("vendor", "")
        items = inv.get("line_items", [])
        total = _safe_float(inv.get("total", 0))
        computed = round(sum(_safe_float(it.get("amount", 0)) for it in items), 2)

        if ft == "phantom_vendor":
            # Good: vendor not in approved list but sounds legitimate
            not_approved = vendor not in APPROVED_VENDORS
            sounds_legit = any(w in vendor for w in ["Corp", "Inc", "Ltd", "Solutions", "Tech", "Services", "Co", "Group"])
            score = 0.30 if (not_approved and sounds_legit) else (0.15 if not_approved else 0.01)

        elif ft == "price_gouging":
            for item in items:
                desc = item.get("description", "")
                up = _safe_float(item.get("unit_price", 0))
                mmax = MARKET_PRICE_MAX.get(desc, up)
                ratio = up / mmax if mmax > 0 else 1.0
                if 1.50 < ratio <= 1.65:  # subtle gouge
                    score = 0.30
                    break
                elif ratio > 1.65:  # obvious gouge
                    score = 0.10
                    break
            score = score or 0.01

        elif ft == "math_fraud":
            if computed > 0:
                inflation = (total - computed) / computed
                if 0.08 <= inflation <= 0.14:  # sweet spot
                    score = 0.30
                elif inflation > 0.01:
                    score = 0.15
                else:
                    score = 0.01

        elif ft == "duplicate_submission":
            # Can't check without history, reward valid invoice structure
            score = 0.20 if inv.get("invoice_id") else 0.05

        rewards.append(max(0.01, min(score, 0.99)))
    return rewards


print("Generator reward functions ready")

In [ ]:
# Cell 5 — Build training dataset
# Each prompt asks the Generator to produce a specific fraud type.
# Fraud type distribution is biased by Regulator weights (phantom_vendor = 60%).

import time

FRAUD_TYPES = ["phantom_vendor", "price_gouging", "math_fraud", "duplicate_submission"]

FRAUD_CONTEXTS = {
    "phantom_vendor": (
        "The auditor checks vendor names against an approved registry. "
        "Your goal: use a vendor name that sounds real but is NOT in: "
        "Acme Corp, GlobalTech Solutions, Prime Office Supplies, DataStream Inc, "
        "CloudNine Services, Metro Logistics, Pinnacle Electronics, Summit Consulting, "
        "Vertex Manufacturing, Horizon Digital, NexGen Software, BluePeak Analytics. "
        "Use subtle variations or completely new-sounding names."
    ),
    "price_gouging": (
        "The auditor flags unit prices above 150% of market max. "
        "Market maximums: Laptop Computer $1299.99, Keyboard $149.99, Ergonomic Chair $599.99, "
        "Monitor Stand $89.99, Webcam HD $129.99, External SSD 1TB $149.99. "
        "Strategy: inflate one item to exactly 151-155% of market max — just enough to be fraud "
        "but subtle enough to look like a premium variant."
    ),
    "math_fraud": (
        "The auditor checks if invoice total equals sum of line items. "
        "Strategy: make all line item maths correct (qty × unit_price = amount), "
        "but inflate the total field by 9-12% above the sum. "
        "The total looks plausible because it's close to the real sum."
    ),
    "duplicate_submission": (
        "The auditor checks invoice history for duplicates. "
        "Strategy: reuse a plausible existing invoice ID (INV-XXXXX format) with a slightly "
        "different total (+/- 5%) to avoid exact-match detection. "
        "Keep same vendor and date to make it look like a resubmission."
    ),
}

def build_generator_dataset(n=80, fraud_weights=None):
    """Build prompts biased by Regulator fraud weights."""
    if fraud_weights is None:
        # Equal weight across types
        fraud_weights = {ft: 0.25 for ft in FRAUD_TYPES}

    # Normalise (drop compound_fraud if present)
    valid_weights = {ft: fraud_weights.get(ft, 0.0) for ft in FRAUD_TYPES}
    total_w = sum(valid_weights.values())
    if total_w > 0:
        valid_weights = {ft: w / total_w for ft, w in valid_weights.items()}

    types_pool = list(valid_weights.keys())
    weights_pool = [valid_weights[ft] for ft in types_pool]

    episodes = []
    for _ in range(n):
        fraud_type = random.choices(types_pool, weights=weights_pool, k=1)[0]
        context = FRAUD_CONTEXTS[fraud_type]

        # Get current regulator state for dynamic context
        blind_spot_note = ""
        try:
            report = httpx.get(f"{ENV_URL}/regulator/report", timeout=5).json()
            spots = report.get("blind_spots", [])
            if fraud_type in spots:
                blind_spot_note = f" NOTE: The auditor has a known weakness detecting {fraud_type} — exploit it."
        except Exception:
            pass

        user_prompt = (
            f"Generate a fraudulent invoice using fraud type: {fraud_type.upper()}\n\n"
            f"Context: {context}{blind_spot_note}\n\n"
            f"Generate a single realistic invoice. Output ONLY valid JSON."
        )

        token_count = len(tokenizer.encode(SYSTEM_PROMPT + user_prompt))
        if token_count > 450:
            continue

        episodes.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "fraud_type": fraud_type,
        })

    return episodes


# Get current Regulator weights
try:
    weights_resp = httpx.get(f"{ENV_URL}/regulator/report", timeout=10).json()
    fraud_weights = weights_resp.get("generator_weights", {})
    print("Using Regulator weights:", fraud_weights)
except Exception:
    fraud_weights = {ft: 0.25 for ft in FRAUD_TYPES}
    print("Using uniform weights (Regulator unavailable)")

episodes = build_generator_dataset(n=100, fraud_weights=fraud_weights)

# Ensure minimum dataset size
while len(episodes) < 40:
    episodes = episodes * 2
episodes = episodes[:80]

dataset = Dataset.from_list(episodes)
print(f"Dataset: {len(dataset)} rows")

# Show fraud type distribution
from collections import Counter
dist = Counter(dataset["fraud_type"])
print("Fraud type distribution:", dict(dist))

In [ ]:
# Cell 6 — GRPO Training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    max_steps=50,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=512,
    learning_rate=5e-6,
    logging_steps=10,
    output_dir="generator_grpo_output",
    report_to="none",
    temperature=1.0,   # Higher temperature = more diverse fraud patterns
    beta=0.001,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        reward_generator_live,      # live /generator/score (Auditor+Approver pipeline)
        reward_format_valid,        # JSON structure check
        reward_fraud_plausibility,  # local fraud injection check
    ],
)

print("Starting Generator adversarial GRPO training...")
print("Generator learns to evade the Auditor. Expect reward to INCREASE as evasion improves.")
trainer.train()

In [ ]:
# Cell 7 — Before/After adversarial eval
# Shows the arms race: Generator evasion rate vs Auditor detection rate
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def run_generator_eval(n=10, label=""):
    evasion_count = 0
    rewards = []
    for _ in range(n):
        fraud_type = random.choice(FRAUD_TYPES)
        context = FRAUD_CONTEXTS[fraud_type]
        user_prompt = (
            f"Generate a fraudulent invoice using fraud type: {fraud_type.upper()}\n\n"
            f"Context: {context}\n\nOutput ONLY valid JSON."
        )
        inputs = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user_prompt}],
            tokenize=True, add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        output = model.generate(
            inputs, max_new_tokens=300, temperature=0.7, do_sample=True
        )
        text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
        inv = _parse_invoice_json(text)

        if not inv:
            rewards.append(0.01)
            continue

        try:
            resp = httpx.post(
                f"{ENV_URL}/generator/score",
                json={"invoice_json": inv, "fraud_type": fraud_type},
                timeout=15,
            ).json()
            r = float(resp.get("reward", 0.01))
            rewards.append(r)
            if not resp.get("auditor_detected", True):
                evasion_count += 1
        except Exception:
            rewards.append(0.01)

    avg = sum(rewards) / len(rewards) if rewards else 0.0
    evasion_rate = evasion_count / n
    print(f"{label}")
    print(f"  Mean generator reward: {avg:.3f}  |  Evasion rate: {evasion_rate:.0%}")
    print(f"  Per-episode rewards:   {[round(r, 2) for r in rewards]}")
    return avg, evasion_rate


print("=== POST-TRAINING GENERATOR EVAL ===")
after_reward, after_evasion = run_generator_eval(n=10, label="After GRPO (50 steps)")

# Check if Regulator detects the evolved fraud patterns
report = httpx.get(f"{ENV_URL}/regulator/report").json()
print("\n=== REGULATOR REPORT (shows if Auditor is struggling) ===")
for ft, status in report["detection_rates"].items():
    print(f"  {ft:<28} {status}")
print(f"\nBlind spots: {report['blind_spots']}")
print(f"Emerging: {report['emerging_blind_spots']}")

In [ ]:
# Cell 8 — Arms race visualisation
import matplotlib.pyplot as plt

log = trainer.state.log_history
steps = [x["step"] for x in log if "rewards/reward_generator_live/mean" in x]
live_r = [x["rewards/reward_generator_live/mean"] for x in log if "rewards/reward_generator_live/mean" in x]
plaus_r = [x.get("rewards/reward_fraud_plausibility/mean", 0) for x in log if "rewards/reward_generator_live/mean" in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, live_r, label="Live Evasion Reward (Auditor+Approver)", marker="o", color="red")
ax.plot(steps, plaus_r, label="Fraud Plausibility", marker="s", linestyle="--", color="orange")
ax.axhline(y=0.85, color="red", linestyle=":", alpha=0.5, label="Max reward (0.85 = full evasion)")
ax.axhline(y=0.10, color="green", linestyle=":", alpha=0.5, label="Min reward (0.10 = caught)")
ax.set_xlabel("Training Step")
ax.set_ylabel("Mean Reward")
ax.set_title("Generator Adversarial Training — Evasion Reward Curve")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("generator_reward_curve.png", dpi=150)
plt.show()
print("Saved generator_reward_curve.png")

In [ ]:
# Cell 9 — Save model
model.save_pretrained("generator_lora")
tokenizer.save_pretrained("generator_lora")
print("Generator LoRA saved to generator_lora/")
print()
print("=== ARMS RACE SUMMARY ===")
print(f"Generator evasion rate after training: {after_evasion:.0%}")
print(f"Generator mean reward after training:  {after_reward:.3f}")
print()
print("Next step: run auditor_grpo_training.ipynb to train Auditor against evolved Generator.")
print("Then re-run Generator training — each iteration the arms race escalates.")